In [108]:
import cv2
import os
import cv2
import mediapipe as mp
import os
import numpy as np
import math

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module='google.protobuf')

In [182]:
## SEGMENTATION DE VIDEOS EN FRAME

# Chemin de la vidéo et du dossier de sortie
video_path = './ressources/noura_squat.mp4'
output_folder = './ressources/noura_squat'

# Créer le dossier de sortie s'il n'existe pas
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Charger la video
cap = cv2.VideoCapture(video_path)

# Vérifier si la vidéo a bien été chargée
if not cap.isOpened():
    print("Erreur : Impossible de charger la vidéo.")
    exit()

# Paramètres pour extraire les frames
frame_rate = cap.get(cv2.CAP_PROP_FPS)  # Obtenir le frame rate (nombre de frames par seconde)
ms_per_frame = 1000 / frame_rate  # Durée en millisecondes de chaque frame
interval_time_ms = 200  # Temps en millisecondes entre chaque frame à enregistrer

time_accumulated = 0  # Accumuler le temps écoulé en millisecondes
frame_count = 0
saved_frame_count = 1

while True:
    # Lire une frame de la vidéo
    ret, frame = cap.read()

    # Vérifier si la frame a bien été lue
    if not ret:
        break

    # Accumuler le temps écoulé
    time_accumulated += ms_per_frame

    # Vérifier si on a dépassé l'intervalle de temps de 500 ms
    if time_accumulated >= interval_time_ms:
        frame_filename = os.path.join(output_folder, f"frame_{saved_frame_count}.jpg")
        cv2.imwrite(frame_filename, frame)
        print(f"Frame sauvegardée : {frame_filename}")
        saved_frame_count += 1
        time_accumulated = 0  # Réinitialiser l'accumulation de temps

    # Incrémenter le compteur de frames
    frame_count += 1

# Libérer la vidéo
cap.release()
print("Traitement terminé.")


Frame sauvegardée : ./ressources/noura_squat\frame_1.jpg
Frame sauvegardée : ./ressources/noura_squat\frame_2.jpg
Frame sauvegardée : ./ressources/noura_squat\frame_3.jpg
Frame sauvegardée : ./ressources/noura_squat\frame_4.jpg
Frame sauvegardée : ./ressources/noura_squat\frame_5.jpg
Frame sauvegardée : ./ressources/noura_squat\frame_6.jpg
Frame sauvegardée : ./ressources/noura_squat\frame_7.jpg
Frame sauvegardée : ./ressources/noura_squat\frame_8.jpg
Frame sauvegardée : ./ressources/noura_squat\frame_9.jpg
Frame sauvegardée : ./ressources/noura_squat\frame_10.jpg
Frame sauvegardée : ./ressources/noura_squat\frame_11.jpg
Frame sauvegardée : ./ressources/noura_squat\frame_12.jpg
Frame sauvegardée : ./ressources/noura_squat\frame_13.jpg
Frame sauvegardée : ./ressources/noura_squat\frame_14.jpg
Frame sauvegardée : ./ressources/noura_squat\frame_15.jpg
Frame sauvegardée : ./ressources/noura_squat\frame_16.jpg
Frame sauvegardée : ./ressources/noura_squat\frame_17.jpg
Frame sauvegardée : ./r

In [25]:
# Initialiser MediaPipe Pose et Drawing
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

# Liste des noms des points clés (PoseLandmark)
landmark_names = {
    mp_pose.PoseLandmark.NOSE: "Nez",
    mp_pose.PoseLandmark.LEFT_EYE_INNER: "Oeil gauche (interieur)",
    mp_pose.PoseLandmark.LEFT_EYE: "Oeil gauche",
    mp_pose.PoseLandmark.LEFT_EYE_OUTER: "Oeil gauche (exterieur)",
    mp_pose.PoseLandmark.RIGHT_EYE_INNER: "Oeil droit (interieur)",
    mp_pose.PoseLandmark.RIGHT_EYE: "Oeil droit",
    mp_pose.PoseLandmark.RIGHT_EYE_OUTER: "Oeil droit (exterieur)",
    mp_pose.PoseLandmark.LEFT_EAR: "Oreille gauche",
    mp_pose.PoseLandmark.RIGHT_EAR: "Oreille droite",
    mp_pose.PoseLandmark.MOUTH_LEFT: "Coin de la bouche gauche",
    mp_pose.PoseLandmark.MOUTH_RIGHT: "Coin de la bouche droite",
    mp_pose.PoseLandmark.LEFT_SHOULDER: "Epaule gauche",
    mp_pose.PoseLandmark.RIGHT_SHOULDER: "Epaule droite",
    mp_pose.PoseLandmark.LEFT_ELBOW: "Coude gauche",
    mp_pose.PoseLandmark.RIGHT_ELBOW: "Coude droit",
    mp_pose.PoseLandmark.LEFT_WRIST: "Poignet gauche",
    mp_pose.PoseLandmark.RIGHT_WRIST: "Poignet droit",
    mp_pose.PoseLandmark.LEFT_HIP: "Hanche gauche",
    mp_pose.PoseLandmark.RIGHT_HIP: "Hanche droite",
    mp_pose.PoseLandmark.LEFT_KNEE: "Genou gauche",
    mp_pose.PoseLandmark.RIGHT_KNEE: "Genou droit",
    mp_pose.PoseLandmark.LEFT_ANKLE: "Cheville gauche",
    mp_pose.PoseLandmark.RIGHT_ANKLE: "Cheville droite",
    mp_pose.PoseLandmark.LEFT_FOOT_INDEX: "Orteil gauche",
    mp_pose.PoseLandmark.RIGHT_FOOT_INDEX: "Orteil droit"
}

def load_image(image_path):
    """Charge une image depuis un chemin spécifié et la convertit en RGB."""
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError(f"Erreur : impossible de charger l'image {image_path}")
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB), image

def detect_pose(image_rgb):
    """Détecte les poses humaines sur une image en utilisant MediaPipe."""
    with mp_pose.Pose() as pose:
        return pose.process(image_rgb)

def annotate_image(image, results):
    """Ajoute des points clés et des labels sur l'image en fonction des résultats de la détection."""
    if results.pose_landmarks:
        h, w, _ = image.shape
        # Dessiner les points clés
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        
        # Ajouter les labels à côté des points
        for idx, landmark in enumerate(results.pose_landmarks.landmark):
            x, y = int(landmark.x * w), int(landmark.y * h)
            if mp_pose.PoseLandmark(idx) in landmark_names:
                label = landmark_names[mp_pose.PoseLandmark(idx)]
                cv2.putText(image, label, (x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2, cv2.LINE_AA)
    return image

def extract_landmarks(results, image):
    """Extrait les coordonnées des points clés sous forme de dictionnaire {partie_du_corps: {x, y}}."""
    landmarks_dict = {}
    if results.pose_landmarks:
        h, w, _ = image.shape
        for idx, landmark in enumerate(results.pose_landmarks.landmark):
            if mp_pose.PoseLandmark(idx) in landmark_names:
                part_name = landmark_names[mp_pose.PoseLandmark(idx)]
                landmarks_dict[part_name] = {"x": landmark.x * w, "y": landmark.y * h}
    return landmarks_dict

def save_image(image, output_path):
    """Enregistre l'image annotée à l'emplacement spécifié."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    cv2.imwrite(output_path, image)

def process_and_save_pose_image(image_path, output_path, save_images=True):
    """Pipeline complet pour charger, traiter, annoter et sauvegarder une image avec la reconnaissance de poses.
       Renvoie également un dictionnaire avec les coordonnées des points clés."""
    image_rgb, image = load_image(image_path)
    results = detect_pose(image_rgb)
    landmarks_dict = extract_landmarks(results, image)
    
    if save_images:
        annotated_image = annotate_image(image, results)
        save_image(annotated_image, output_path)

    return landmarks_dict

def process_images_in_folder(input_folder, output_folder, save_images=True):
    """Applique la fonction process_and_save_pose_image à toutes les images d'un dossier.
       Retourne un tableau de dictionnaires contenant les points clés pour chaque image."""
    landmarks_list = []
    
    # Parcourir tous les fichiers du dossier
    for filename in os.listdir(input_folder):
        # Vérifier si le fichier est une image (tu peux étendre cette liste à d'autres formats si nécessaire)
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            input_path = os.path.join(input_folder, filename)
            output_path = os.path.join(output_folder, filename)
                        
            # Appliquer la fonction process_and_save_pose_image à chaque image et récupérer les landmarks
            try:
                landmarks_dict = process_and_save_pose_image(input_path, output_path, save_images)
                landmarks_list.append(landmarks_dict)
            except Exception as e:
                print(f"Erreur lors du traitement de {filename}: {e}")
    
    return landmarks_list

def afficher_positions(landmarks_dict):
    """Affiche les coordonnées (x, y) de chaque partie du corps dans le dictionnaire."""
    for part, coordinates in landmarks_dict.items():
        x = coordinates['x']
        y = coordinates['y']
        print(f"{part}: x = {x:.2f}, y = {y:.2f}")


In [5]:
# Exemple d'utilisation sur une image
image_path = './img/cedric.jpg'
output_path = './output_img/image_labellisee.jpg'
pos = process_and_save_pose_image(image_path, output_path, save_images=True)
afficher_positions(pos)

Nez: x = 1675.73, y = 2138.95
Oeil gauche (interieur): x = 1706.68, y = 2092.64
Oeil gauche: x = 1726.31, y = 2092.41
Oeil gauche (exterieur): x = 1743.94, y = 2092.41
Oeil droit (interieur): x = 1646.49, y = 2096.53
Oeil droit: x = 1625.98, y = 2098.52
Oeil droit (exterieur): x = 1607.05, y = 2100.01
Oreille gauche: x = 1776.01, y = 2119.59
Oreille droite: x = 1585.26, y = 2124.29
Coin de la bouche gauche: x = 1720.40, y = 2197.74
Coin de la bouche droite: x = 1643.83, y = 2200.30
Epaule gauche: x = 1956.45, y = 2405.24
Epaule droite: x = 1439.66, y = 2394.22
Coude gauche: x = 2337.66, y = 2528.22
Coude droit: x = 1100.16, y = 2475.92
Poignet gauche: x = 2707.17, y = 2572.25
Poignet droit: x = 704.89, y = 2516.64
Hanche gauche: x = 1805.92, y = 3217.18
Hanche droite: x = 1518.53, y = 3210.25
Genou gauche: x = 1854.00, y = 3771.04
Genou droit: x = 1473.04, y = 3765.26
Cheville gauche: x = 1923.86, y = 4309.82
Cheville droite: x = 1416.03, y = 4308.61
Orteil gauche: x = 2040.87, y = 450

In [183]:
# Exemple d'utilisation sur un dossier
# Le modele produit 25 points à chaque fois
tab = []

input_folder = './ressources/noura_squat'
output_folder = './ressources/noura_squat_labelled'
positions = process_images_in_folder(input_folder, output_folder, save_images=True)
for index, position in enumerate(positions, start=1):
    print(f"Position {index}:")

    # Vérifier que le dictionnaire a le nombre d'éléments attendu
    afficher_positions(position)
    tab.append(len(position))
    
    print() 

print("Nombre de frames :", len(tab))
print(tab)

Position 1:
Nez: x = 242.68, y = 306.63
Oeil gauche (interieur): x = 249.61, y = 290.53
Oeil gauche: x = 253.94, y = 290.21
Oeil gauche (exterieur): x = 258.23, y = 289.96
Oeil droit (interieur): x = 231.65, y = 292.00
Oeil droit: x = 224.18, y = 292.50
Oeil droit (exterieur): x = 217.73, y = 292.86
Oreille gauche: x = 262.78, y = 293.73
Oreille droite: x = 207.34, y = 297.61
Coin de la bouche gauche: x = 250.90, y = 321.18
Coin de la bouche droite: x = 228.91, y = 323.31
Epaule gauche: x = 296.83, y = 355.56
Epaule droite: x = 169.52, y = 366.21
Coude gauche: x = 335.08, y = 425.72
Coude droit: x = 129.13, y = 454.33
Poignet gauche: x = 371.92, y = 466.96
Poignet droit: x = 112.05, y = 474.93
Hanche gauche: x = 279.70, y = 507.76
Hanche droite: x = 203.85, y = 507.93
Genou gauche: x = 292.89, y = 624.98
Genou droit: x = 218.36, y = 619.28
Cheville gauche: x = 296.80, y = 750.63
Cheville droite: x = 222.37, y = 753.92
Orteil gauche: x = 302.94, y = 813.78
Orteil droit: x = 228.63, y = 

In [113]:
# Test Biceps Curls
# Utilisation du poignet droit, coude droit, epaule droite, poignet gauche, coude gauche, epaule gauche

# Les frames vont arriver 1 à 1, il faudrait qu'on analyse aussi la vitesse du mouvement également pour savoir à peu près si le mouvement est bien fait.
# Il faut analyser l'angle, la vitesse, la position des points clés pour savoir si le mouvement est bien fait.

# Pour vérifier qu'un curl biceps est bien fait, il faut que : Le corps soit bien droit, épaule alignée, coude aligné, poignet aligné
# Il faut que le coude soit entre l'épaule et les hanches

def calculate_angle(p1, p2, p3):     
    a = np.array([p1['x'], p1['y']])
    b = np.array([p2['x'], p2['y']])
    c = np.array([p3['x'], p3['y']])
    
    ba = a - b
    bc = c - b
    
    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
    angle = np.arccos(cosine_angle)
    
    angle = np.degrees(angle)
    return angle


interval_time_ms = 200  # Temps en millisecondes entre chaque frame

def calculate_angle_speed(angle1, angle2, step):
    angle_speed = (angle2 - angle1) / step
    return angle_speed

In [53]:
# Test calculate angle
# Il faut imiter le fait que la frame arrive

nb_frames = 78
tab_angle_droit = []
tab_angle_gauche = []
tab_angle_speed_droit = []
tab_angle_speed_gauche = []

for i in range(nb_frames):
    img_path = './ressources/cedric_squat/frame_' + str(i+1) + '.jpg'
    pos = process_and_save_pose_image(img_path, "", save_images=False)
    tab_angle_droit.append(calculate_angle(pos['Poignet droit'], pos['Coude droit'], pos['Epaule droite']))
    tab_angle_gauche.append(calculate_angle(pos['Poignet gauche'], pos['Coude gauche'], pos['Epaule gauche']))
    
    if i > 0:
        tab_angle_speed_droit.append(calculate_angle_speed(tab_angle_droit[i-1], tab_angle_droit[i], interval_time_ms))
        tab_angle_speed_gauche.append(calculate_angle_speed(tab_angle_gauche[i-1], tab_angle_gauche[i], interval_time_ms))
        print("Frame", i)
        print("Angle droit :", tab_angle_droit[i], "Angle gauche :", tab_angle_gauche[i])
        print("Vitesse angle droit :", calculate_angle_speed(tab_angle_droit[i-1], tab_angle_droit[i], interval_time_ms), "Vitesse angle gauche :", calculate_angle_speed(tab_angle_gauche[i-1], tab_angle_gauche[i], interval_time_ms))
        print()

    else:
        print("Frame", i)
        print("Angle droit :", tab_angle_droit[i], "Angle gauche :", tab_angle_gauche[i])
        print()


# Il me faudrait une limite angle genre 150° et 20° pour valider le curl biceps
# J'active quand je suis à 150° +, puis tant que la vitesse est négative et que sa valeur absolue est en dessous d'une certaine valeur, je continue, dès que la vitesse change de signe, je check si l'angle est en dessous de 20°, si c'est le cas, je valide le mouvement.

Frame 0
Angle droit : 166.82682730368833 Angle gauche : 178.75002011946873

Frame 1
Angle droit : 96.20178023117266 Angle gauche : 169.4215212580856
Vitesse angle droit : -0.35312523536257834 Vitesse angle gauche : -0.046642494306915694

Frame 2
Angle droit : 165.23663172227424 Angle gauche : 53.01292771984302
Vitesse angle droit : 0.3451742574555079 Vitesse angle gauche : -0.5820429676912129

Frame 3
Angle droit : 158.82429182758025 Angle gauche : 177.38197853508214
Vitesse angle droit : -0.03206169947346993 Vitesse angle gauche : 0.6218452540761956

Frame 4
Angle droit : 167.8534053302755 Angle gauche : 157.86137256538905
Vitesse angle droit : 0.045145567513476266 Vitesse angle gauche : -0.09760302984846547

Frame 5
Angle droit : 163.18307291460843 Angle gauche : 144.1026155685942
Vitesse angle droit : -0.02335166207833538 Vitesse angle gauche : -0.06879378498397429

Frame 6
Angle droit : 169.87469158361498 Angle gauche : 157.51068770890186
Vitesse angle droit : 0.03345809334503272 V

KeyboardInterrupt: 

In [9]:
nbr_curl_biceps_done = 0
number_of_frames_checked = 0
interval_time_ms = 200  # Temps en millisecondes entre chaque frame
move_state = -1

def check_curl_biceps(frame):
    global nbr_curl_biceps_done
    global number_of_frames_checked
    global move_state

    pos = process_and_save_pose_image(frame, "", save_images=False)
    angle_droit = calculate_angle(pos['Poignet droit'], pos['Coude droit'], pos['Epaule droite'])
    angle_gauche = calculate_angle(pos['Poignet gauche'], pos['Coude gauche'], pos['Epaule gauche'])
    number_of_frames_checked += 1

    if move_state == -1:  
        if angle_droit > 150 and angle_gauche > 150:
            move_state = 0 
        return

    if move_state == 0:
        if angle_droit < 30 and angle_gauche < 30: 
            move_state = 1
        return

    elif move_state == 1:
        if angle_droit > 150 and angle_gauche > 150:
            if number_of_frames_checked * interval_time_ms > 1500:
                nbr_curl_biceps_done += 1  

            move_state = 0
            number_of_frames_checked = 0

In [179]:
nbr_frames = 85
frames = []
for i in range(nbr_frames):
    frames.append('./ressources/noura_squat/frame_' + str(i+1) + '.jpg')

In [19]:
# Il faut faire essayer de faire attention au temps que prend l'exécution de la fonction check_curl_biceps
nbr_curl_biceps_done = 0

j = 0
for frame in frames:
    check_curl_biceps(frame)
    print("Frame", j)
    print("Nombre de curl biceps effectués :", nbr_curl_biceps_done)
    print()
    j += 1

Frame 0
Nombre de curl biceps effectués : 0

Frame 1
Nombre de curl biceps effectués : 0

Frame 2
Nombre de curl biceps effectués : 0

Frame 3
Nombre de curl biceps effectués : 0

Frame 4
Nombre de curl biceps effectués : 0

Frame 5
Nombre de curl biceps effectués : 0

Frame 6
Nombre de curl biceps effectués : 0

Frame 7
Nombre de curl biceps effectués : 0

Frame 8
Nombre de curl biceps effectués : 0

Frame 9
Nombre de curl biceps effectués : 0

Frame 10
Nombre de curl biceps effectués : 0

Frame 11
Nombre de curl biceps effectués : 0

Frame 12
Nombre de curl biceps effectués : 0

Frame 13
Nombre de curl biceps effectués : 0



KeyboardInterrupt: 

# Detect squats

In [186]:
def is_body_straight(curr_landmarks, tolerance=0.1):
    required_points = ['Epaule droite', 'Hanche droite', 'Genou droit', 'Cheville droite']

    for point in required_points:
        if point not in curr_landmarks:
            raise ValueError(f"Le point clé '{point}' est manquant dans les landmarks.")
        
    angles = []

    angles.append(calculate_angle(curr_landmarks['Epaule droite'], curr_landmarks['Hanche droite'], curr_landmarks['Genou droit']))
    angles.append(calculate_angle(curr_landmarks['Hanche droite'], curr_landmarks['Genou droit'], curr_landmarks['Cheville droite']))

    for angle in angles:
        if not (180 * (1 - tolerance) <= angle <= 180 * (1 + tolerance)):
            return False  

    return True

def is_legs_straight(curr_landmarks, tolerance=0.1):
    required_points = ['Hanche droite', 'Genou droit', 'Cheville droite']

    for point in required_points:
        if point not in curr_landmarks:
            raise ValueError(f"Le point clé '{point}' est manquant dans les landmarks.")
        
    angle = calculate_angle(curr_landmarks['Hanche droite'], curr_landmarks['Genou droit'], curr_landmarks['Cheville droite'])

    if not (180 * (1 - tolerance) <= angle <= 180 * (1 + tolerance)):
        return False  
    return True

def calculate_distance(p1, p2):
    return np.sqrt((p1['x'] - p2['x'])**2 + (p1['y'] - p2['y'])**2)

def calculate_height(curr_landmarks, required_points=['Epaule droite', 'Hanche droite', 'Genou droit', 'Cheville droite']):
    for point in required_points:
        if point not in curr_landmarks:
            raise ValueError(f"Le point clé '{point}' est manquant dans les landmarks.")
        
    total_distance = 0
    for i in range(len(required_points) - 1):
        point1 = curr_landmarks[required_points[i]]
        point2 = curr_landmarks[required_points[i + 1]]
        total_distance += calculate_distance(point1, point2)

    return total_distance

def is_height_valid(height, curr_landmarks, tolerance=0.4):
    current_height = calculate_height(curr_landmarks)
    return abs(current_height - height) / height <= tolerance

def calculate_angle_opposite(a, b, c):
    """
    Calcule l'angle en face du côté 'a' d'un triangle en utilisant la loi des cosinus.
    """
    if a + b <= c or a + c <= b or b + c <= a:
        raise ValueError("Les longueurs fournies ne forment pas un triangle valide.")

    cosine_theta = (b**2 + c**2 - a**2) / (2 * b * c)
    cosine_theta = max(-1, min(1, cosine_theta))
    
    angle_rad = math.acos(cosine_theta)
    angle_deg = math.degrees(angle_rad)
    
    return angle_deg

# Il faut que je regarde la taille au début de la personne
# Ensuite il fait le mouvement de descente, le mouvement de descente est validé seulement si ca fait genre 80% de la taille initiale
# Ensuite il fait le mouvement de remontée jusqu'à que la taille soit égale à la taille initiale et que le corps soit droit, si le corps est droit mais que
# la taille est différente, il faut refaire l'initialisation

# Si tout ca passe, alors je peux valider le mouvement

def compare_segments(segment1, segment2):
    """
    Compare deux segments en pourcentage.
    """
    if segment2 == 0:  # Éviter la division par zéro
        raise ValueError("La longueur du deuxième segment ne peut pas être zéro.")

    percentage = (segment1 / segment2) * 100
    return percentage


legs_height = 0
number_of_frames_checked = 0
squat_state = -1 # -1 = down, 1 = up

def check_squat(frame):
    global legs_height
    global number_of_frames_checked
    global squat_state

    curr_landmarks = process_and_save_pose_image(frame, "", save_images=False)
    number_of_frames_checked += 1

    if legs_height == 0:
        if is_body_straight(curr_landmarks):
            legs_height= calculate_height(curr_landmarks, ['Hanche droite', 'Genou droit', 'Cheville droite'])
            squat_state = -1
            print("Legs Hauteur initialisée à", legs_height)
            return False
        else:
            print("Le corps n'est pas droit, l'initialisation de la hauteur est impossible.")
            return False
    else:
        percentage = compare_segments(calculate_height(curr_landmarks, ['Hanche droite', 'Genou droit', 'Cheville droite']), legs_height)
        
        if squat_state == -1:  
            if percentage < 65:
                squat_state = 1
                print("Changement d'état en up")
            return False
        
        if squat_state == 1:
            if percentage > 95:
                if number_of_frames_checked * interval_time_ms > 1500:
                    squat_state = -1
                    print("Changement d'état en down")
                    number_of_frames_checked = 0
                    return True
            else:
                return False
    
    return False

In [187]:
# Pour reconnaitre un squat il faut que je vérifie que tout son corps descend bien, et que je vérifie que l'angle de ses genoux descend
# jusqu'à un certain degré, et ensuite je dois vérifier que ca remonte

# Donc ce qui est important c'est vraiment le torse et les genoux
# Pour les genoux

# En fait il faut que je calcule la distance 
# Il me faut une fonction qui permet de voir si le corps est bien droit
# Il faut que je regarde au début si le corps est droit, et si la taille correspond à la taille de la personne au début de l'exercice 
# Si la taille est plus petite ou plus grande, il faut recalibrer la taille, on attend que le corps soit à nouveau droit
# Si le corps est bien à sa taille normale, on initialise l'état, et on commence à vérifier si le squat est bien fait, en regardant juste la distance entre le
# hanche et le pied, car en utilisant la loi des cosinus on peut approximer l'angle entre pied genou et hanche

# Test body straight
def test1(frames):
    j = 1
    for frame in frames:
        curr_landmarks = process_and_save_pose_image(frame, "", save_images=False)
        print("Frame", j)
        if is_body_straight(curr_landmarks=curr_landmarks):
            print("Le corps est droit")
        else:
            print("Le corps n'est pas droit")
        j += 1

# Test height + body straight
def test2(frames):
    height = 0
    j = 1
    for frame in frames:
        print("Frame", j)
        curr_landmarks = process_and_save_pose_image(frame, "", save_images=False)
        if height == 0:
            if is_body_straight(curr_landmarks):
                height = calculate_height(curr_landmarks)
                print("Hauteur initialisée à", height)
        else:
            tolerance = 0.2 if is_body_straight(curr_landmarks, tolerance=0.1) else 0.4
            if not is_height_valid(height, curr_landmarks, tolerance = tolerance):
                if is_body_straight(curr_landmarks):
                    print("La taille n'est pas valide, mais le corps est droit.")
                else:
                    print("La taille n'est pas valide et le corps n'est pas droit.")
        
        j += 1

# Test calcul d'angle avec loi des cosinus hanche-genou-cheville
def test3(frames):
    j = 1
    for frame in frames:
        curr_landmarks = process_and_save_pose_image(frame, "", save_images=False)
        print("Frame", j)
        a = calculate_distance(curr_landmarks['Hanche droite'], curr_landmarks['Cheville droite'])
        b = calculate_distance(curr_landmarks['Hanche droite'], curr_landmarks['Genou droit'])
        c = calculate_distance(curr_landmarks['Genou droit'], curr_landmarks['Cheville droite'])

        
        opposite_angle = calculate_angle_opposite(a, b, c)
        normal_angle = calculate_angle(curr_landmarks['Hanche droite'], curr_landmarks['Genou droit'], curr_landmarks['Cheville droite'])

        print("Angle hanche-genou-cheville :", opposite_angle)
        print("Angle hanche-genou-cheville normal :", normal_angle)
        j += 1

def test4(frames):
    """
    Recherche les minimums locaux d'angles pour les côtés droit et gauche et les maximums locaux d'angles.
    Affiche les résultats à des frames spécifiques.
    """
    # Frames d'arrêt pour vérifier les minimums locaux
    # Cedric
    # arret = [12, 23, 36, 47, 60, 72, 84]
    # Lou
    arret = [12, 22, 32, 42, 55, 69, 79]

    # Initialisation des minimums pour les angles droit et gauche
    min_angles_droit = [400] * len(arret)
    min_frame_index_droit = [0] * len(arret)
    
    min_angles_gauche = [400] * len(arret)
    min_frame_index_gauche = [0] * len(arret)

    j = 1  # Index global des frames
    arret_index = 0  # Index pour suivre les étapes d'arrêt

    for frame in frames:
        # Extraction des landmarks pour la frame actuelle
        curr_landmarks = process_and_save_pose_image(frame, "", save_images=False)
        
        # Calcul des angles pour le côté droit et le côté gauche
        normal_angle_droit = calculate_angle(
            curr_landmarks['Hanche droite'], curr_landmarks['Genou droit'], curr_landmarks['Cheville droite']
        )
        normal_angle_gauche = calculate_angle(
            curr_landmarks['Hanche gauche'], curr_landmarks['Genou gauche'], curr_landmarks['Cheville gauche']
        )

        # Mise à jour du minimum pour l'angle droit
        if normal_angle_droit < min_angles_droit[arret_index]:
            min_angles_droit[arret_index] = normal_angle_droit
            min_frame_index_droit[arret_index] = j
        
        # Mise à jour du minimum pour l'angle gauche
        if normal_angle_gauche < min_angles_gauche[arret_index]:
            min_angles_gauche[arret_index] = normal_angle_gauche
            min_frame_index_gauche[arret_index] = j

        j += 1

        # Arrêter et afficher les résultats à des frames spécifiées
        if j == arret[arret_index]:
            print(f"Frame {j}")
            print(f"Angle minimum droit : {min_angles_droit[arret_index]:.2f}° atteint à la frame {min_frame_index_droit[arret_index]}")
            print(f"Angle minimum gauche : {min_angles_gauche[arret_index]:.2f}° atteint à la frame {min_frame_index_gauche[arret_index]}")
            
            arret_index += 1
            if arret_index == len(arret):  # Arrêter si toutes les étapes ont été couvertes
                break

def test4b(frames):
    """
    Recherche les minimums locaux d'angles pour les côtés droit et gauche et les maximums locaux d'angles.
    Affiche les résultats à des frames spécifiques.
    """
    # Frames d'arrêt pour vérifier les minimums locaux
    arret = [12, 23, 36, 47, 60, 72, 84]

    # Initialisation des minimums pour les angles droit et gauche
    min_angles_droit = [0] * len(arret)
    min_frame_index_droit = [0] * len(arret)
    
    min_angles_gauche = [0] * len(arret)
    min_frame_index_gauche = [0] * len(arret)

    j = 1  # Index global des frames
    arret_index = 0  # Index pour suivre les étapes d'arrêt

    for frame in frames:
        # Extraction des landmarks pour la frame actuelle
        curr_landmarks = process_and_save_pose_image(frame, "", save_images=False)
        
        # Calcul des angles pour le côté droit et le côté gauche
        normal_angle_droit = calculate_angle(
            curr_landmarks['Hanche droite'], curr_landmarks['Genou droit'], curr_landmarks['Cheville droite']
        )
        normal_angle_gauche = calculate_angle(
            curr_landmarks['Hanche gauche'], curr_landmarks['Genou gauche'], curr_landmarks['Cheville gauche']
        )

        # Mise à jour du minimum pour l'angle droit
        if normal_angle_droit > min_angles_droit[arret_index]:
            min_angles_droit[arret_index] = normal_angle_droit
            min_frame_index_droit[arret_index] = j
        
        # Mise à jour du minimum pour l'angle gauche
        if normal_angle_gauche > min_angles_gauche[arret_index]:
            min_angles_gauche[arret_index] = normal_angle_gauche
            min_frame_index_gauche[arret_index] = j

        j += 1

        # Arrêter et afficher les résultats à des frames spécifiées
        if j == arret[arret_index]:
            print(f"Frame {j}")
            print(f"Angle maximum droit : {min_angles_droit[arret_index]:.2f}° atteint à la frame {min_frame_index_droit[arret_index]}")
            print(f"Angle maximum gauche : {min_angles_gauche[arret_index]:.2f}° atteint à la frame {min_frame_index_gauche[arret_index]}")
            
            arret_index += 1
            if arret_index == len(arret):  # Arrêter si toutes les étapes ont été couvertes
                break

def test5(frames):
    j = 1
    number_of_squats_done = 0
    for frame in frames:
        print("Frame", j)
        if check_squat(frame):
            number_of_squats_done += 1
            print("Nombre de squats effectués :", number_of_squats_done)
        j += 1

test5(frames)

Frame 1
Legs Hauteur initialisée à 324.51739019694105
Frame 2
Frame 3
Frame 4
Frame 5
Frame 6
Frame 7
Changement d'état en up
Frame 8
Frame 9
Frame 10
Frame 11
Frame 12
Changement d'état en down
Nombre de squats effectués : 1
Frame 13
Frame 14
Frame 15
Frame 16
Frame 17
Frame 18
Frame 19
Changement d'état en up
Frame 20
Frame 21
Frame 22
Frame 23
Changement d'état en down
Nombre de squats effectués : 2
Frame 24
Frame 25
Frame 26
Frame 27
Frame 28
Frame 29
Frame 30
Changement d'état en up
Frame 31
Frame 32
Frame 33
Frame 34
Frame 35
Changement d'état en down
Nombre de squats effectués : 3
Frame 36
Frame 37
Frame 38
Frame 39
Frame 40
Frame 41
Frame 42
Changement d'état en up
Frame 43
Frame 44
Frame 45
Frame 46
Frame 47
Changement d'état en down
Nombre de squats effectués : 4
Frame 48
Frame 49
Frame 50
Frame 51
Frame 52
Frame 53
Frame 54
Frame 55
Changement d'état en up
Frame 56
Frame 57
Frame 58
Frame 59
Frame 60
Changement d'état en down
Nombre de squats effectués : 5
Frame 61
Frame 62
